In [17]:
import os
from dotenv import load_dotenv
load_dotenv()
import groq 
import requests
from minsearch import Index

In [127]:
## Create Groq client
def create_client():
    client=groq.Groq(api_key=os.environ.get("GROQ_API_KEY"))
    return client


In [22]:
 
def get_data():
    index_url="https://datatalks.club/faq/json/courses.json"
    responses=requests.get(url=index_url).json()
    documents=[]
    url_prefix="https://datatalks.club/faq"
    for response in responses:
        course_url=f"{url_prefix}{response['path']}"
        course_data=requests.get(course_url)
        course_data.raise_for_status()
        documents.extend(course_data.json())

    return documents


all_documents=get_data()


In [30]:
all_documents[1]

{'id': '226a4baf2f',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'What’s new in the 2025 edition?',
 'answer': '- Deployment module updated to **FastAPI** (replacing Flask) and new tools.\n- Neural networks taught with **PyTorch** (theory videos in Keras are kept; an additional PyTorch implementation video is provided).\n- Deep learning deployment uses **ONNX Runtime** on AWS Lambda (replacing TensorFlow Lite).'}

In [24]:
index=Index(
    text_fields=["question","section","answer"],
    keyword_fields=["course"]
)
index.fit(all_documents)

In [ ]:
index.search(query="question")

In [54]:
def search(question,course="llm-zoomcamp"):
    return index.search(
        query=question,
        boost_dict={"question":2.0,"answer":1.0,"section":0.5},
        filter_dict={"course":course},
        num_results=5
    )

In [ ]:
search_results=search("Can I join the course")
search_results

In [84]:
def build_context(search_results):
    lines=[]
    for doc in search_results:
        lines.append(doc['section'])
        lines.append("Q: "+doc['question'])
        lines.append("A: "+doc['answer'])
        lines.append("\n")
    return "\n".join(lines).strip()

In [ ]:
context=build_context(search_results)
print(context)

In [116]:
INSTRUCTIONS=""" 
You are a helpful assistant, answer the {question} based on the {context} provided and if the answer is not found in the {context},reply I don't know
"""

In [117]:
USER_PROMPT_TEMPLATE= """ 
Question:{question}
Context:{context}
"""

In [118]:
question="I just discovered the course, Can I still join?"

In [119]:
USER_PROMPT_TEMPLATE.format(question=question,context=context).strip()

'Question:I just discovered the course, Can I still join?\nContext:General Course-Related Questions\nQ: I just discovered the course. Can I still join?\nA: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\n\nGeneral Course-Related Questions\nQ: Does the course certificate show the number of course hours?\nA: No. The certificate does not state a total number of hours.\n\n\nGeneral Course-Related Questions\nQ: Certificate: Can I follow the course in a self-paced mode and get a certificate?\nA: No, you can only get a certificate if you finish the course with a "live" cohort.\n\nTo get the certificate, you need to finish a capstone project and complete the\nrequired peer reviews. Homework is not required. You can work through the\nmaterial and prepare your project in self-paced mode, but project submission and\npeer review must happen while a live cohort is accepting them.\n\n\nGeneral Course-Related Questions\nQ: How

In [121]:
def build_prompt(question,search_results):
    context=build_context(search_results)
    prompt=USER_PROMPT_TEMPLATE.format(question=question,context=context)
    return prompt.strip()

In [ ]:
prompt=build_prompt(question=question,search_results=search_results)
print(prompt)

In [157]:
def call_llm(prompt,instructions):
    client=client=groq.Groq(api_key=os.environ.get("GROQ_API_KEY"))
    response=client.chat.completions.create(messages=[{"role":"system","content":instructions},{"role":"user","content":prompt}],model="qwen/qwen3.6-27b",reasoning_format="hidden")
    return response.choices[0].message.content

In [158]:

llm_response=call_llm(prompt=build_prompt(question=question,search_results=search_results),instructions=INSTRUCTIONS)


In [159]:
llm_response

'Yes, you can still join. However, if you want to receive a certificate, you need to submit your project while the course is still accepting submissions. You can start learning at any time, as the videos and course materials are available immediately.'

In [136]:
print(llm_response)


<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Question:** "I just discovered the course, Can I still join?"
   - **Context:** Provided QA pairs about a course (LLM Zoomcamp).
   - **Constraint:** Answer based *only* on the context. If not found, reply "I don't know".

2.  **Scan Context for Keywords/Concepts:**
   - Keywords: "I just discovered the course", "Can I still join", "join", "discovered"
   - Look at the first QA pair in the context:
     - Q: I just discovered the course. Can I still join?
     - A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

3.  **Extract Answer:**
   - The context directly answers the question.
   - Answer: Yes, but if you want to receive a certificate, you need to submit your project while they are still accepting submissions.

4.  **Format Output:**
   - Keep it concise and directly aligned with the context.
   - Ensure it strictly follows the constrain